# Stage 3 depth/width screen (`gs_s3dw`)

This notebook analyses `configs/gnn_graph_screening/stage3_depth_width`: the
actor-GNN depth-by-width grid derived from the Stage 2 unaugmented bus baseline
`e0n0v0`.

Two factors are screened:

- **depth** -- `gnn_layers` in `{1, 2, 3}` (message-passing rounds);
- **width** -- `gnn_hidden_dim` in `{16, 32, 64, 128}`, with `gnn_out_dim`
  matched to it for a clean width comparison.

That is 12 architectures; the folder declares three seeds each (36 configs).
The baseline is **`mp2_h128`**, which reproduces the Stage 2 baseline depth and
width under the new naming.

Two independent result sources are used:

1. the **training curves** downloaded into `outputs/run_data/gs_s3dw`;
2. the **full-test evaluation** of each run's best checkpoint, over the complete
   held-out test split, which is the statistic the heatmaps and ranking use.

> **Seed coverage.** Whatever subset of seeds has actually been downloaded is
> detected automatically and reported in the coverage section. With a single
> seed per architecture there is no seed uncertainty to show, so the uncertainty
> bands and per-cell spread turn themselves off rather than displaying a
> misleading empty band. Re-running after downloading more seeds enables them
> with no edits.

In [1]:
from pathlib import Path
import importlib
import json
import sys
import tomllib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError(
        "Could not locate Topology_Task/analysis/metrics/helpers"
    )

import wandb_metrics as wm
wm = importlib.reload(wm)
import survival_comparison as sc
sc = importlib.reload(sc)
print("wandb_metrics:", wm.__file__)
print("survival_comparison:", sc.__file__)
print("task directory:", wm.TASK_DIR)
print("done")

wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py
survival_comparison: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/survival_comparison.py
task directory: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
done


## Analysis controls

`FULL_TEST_EVAL_DIR` is searched **recursively**, so it works whether the
results sit directly in `outputs/full_test_eval` or in a dated subfolder such as
`gs_s3dw_s0_full_test_20260803`.

In [2]:
USE_LOCAL_CACHE_ONLY = True
SMOOTH_WINDOW = 5
TARGET_BUDGET_STEPS = 15_000_000
COMPARISON_BUDGET_STEPS = None

S3DW_DIR = (
    wm.TASK_DIR / "configs" / "gnn_graph_screening" / "stage3_depth_width"
)
FULL_TEST_EVAL_DIR = wm.TASK_DIR / "outputs" / "full_test_eval"
FULL_TEST_GLOB = "*gs_s3dw_*.json"
EXPECTED_FULL_TEST_EPISODES = 201

S3DW_RUN_PREFIX = "gs_s3dw_bus_n0_none_e0n0v0"

# Actor parameter count. This is NOT in wandb_metrics.METRICS, the whitelist
# that full_history_to_long() filters against, so it must be added before the
# histories are loaded or the column is silently dropped.
ACTOR_PARAM_METRIC = "model/actor_params"

BASELINE_DEPTH = 2
BASELINE_WIDTH = 128
BASELINE_LABEL = f"mp{BASELINE_DEPTH}_h{BASELINE_WIDTH}"

DEPTH_ORDER = [1, 2, 3]
WIDTH_ORDER = [16, 32, 64, 128]
DEPTH_LABEL_ORDER = [f"mp{value}" for value in DEPTH_ORDER]
WIDTH_LABEL_ORDER = [f"h{value}" for value in WIDTH_ORDER]
ARCHITECTURE_ORDER = [
    f"mp{depth}_h{width}" for depth in DEPTH_ORDER for width in WIDTH_ORDER
]

FACTOR_COLUMNS = ["gnn_layers", "gnn_hidden_dim"]

print("config folder:", S3DW_DIR)
print("full-test results:", FULL_TEST_EVAL_DIR)
print("baseline:", BASELINE_LABEL)
print("done")

config folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_graph_screening/stage3_depth_width
full-test results: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval
baseline: mp2_h128
done


## Build the run catalog from TOML

Depth and width are read from the config files rather than parsed out of the
file names, and the derived `mp{d}_h{w}` code is cross-checked against the name
so a mislabelled config cannot pass silently.

In [3]:
def architecture_record(path):
    with path.open("rb") as file:
        config = tomllib.load(file)
    args = config["args"]
    run = config.get("run", {})
    depth = int(args.get("gnn_layers", 2))
    hidden = int(args.get("gnn_hidden_dim", 128))
    record = {
        "config_path": str(path.relative_to(wm.TASK_DIR)),
        "config": path.name,
        "run_name": str(run.get("name", path.stem)),
        "seed": int(args.get("seed", 0)),
        "declared_cuda": bool(args.get("cuda", False)),
        "graph_type": str(args.get("gnn_graph_type", "bus")),
        "encoder": str(args.get("gnn_type", "gine")),
        "gnn_layers": depth,
        "gnn_hidden_dim": hidden,
        "gnn_out_dim": int(args.get("gnn_out_dim", hidden)),
        "readout": str(args.get("gnn_readout_aggr", "mean")),
        "substation_edges": bool(args.get("gnn_add_substation_edges", False)),
        "substation_nodes": bool(args.get("gnn_add_substation_nodes", False)),
        "configured_steps": int(args.get("total_timesteps", TARGET_BUDGET_STEPS)),
        "time_limit_minutes": float(args.get("time_limit", np.nan)),
    }
    record["depth_label"] = f"mp{depth}"
    record["width_label"] = f"h{hidden}"
    record["architecture"] = f"mp{depth}_h{hidden}"
    record["is_baseline"] = (
        depth == BASELINE_DEPTH and hidden == BASELINE_WIDTH
    )
    return record


if not S3DW_DIR.exists():
    raise FileNotFoundError(f"Missing config folder: {S3DW_DIR}")

config_catalog = pd.DataFrame(
    [architecture_record(path) for path in sorted(S3DW_DIR.glob("*.toml"))]
)
if config_catalog.empty:
    raise RuntimeError(f"No TOML files were found in {S3DW_DIR}.")

# The derived code must match the code embedded in the run name.
name_codes = config_catalog["run_name"].str.extract(r"_(mp\d+_h\d+)_s\d+$")[0]
mismatched = config_catalog[name_codes != config_catalog["architecture"]]
if not mismatched.empty:
    raise ValueError(
        "Config depth/width disagrees with the run name for: "
        + ", ".join(mismatched["run_name"])
    )

print(f"Cataloged {len(config_catalog)} declared runs.")

held_constant = [
    "graph_type",
    "encoder",
    "readout",
    "substation_edges",
    "substation_nodes",
    "configured_steps",
]
for column in held_constant:
    values = sorted(config_catalog[column].astype(str).unique())
    flag = "" if len(values) == 1 else "  <-- NOT CONSTANT"
    print(f"  {column}: {values}{flag}")

if not config_catalog["gnn_out_dim"].eq(config_catalog["gnn_hidden_dim"]).all():
    raise ValueError("gnn_out_dim is not matched to gnn_hidden_dim everywhere.")
print("  gnn_out_dim matches gnn_hidden_dim for every config")

display(
    config_catalog.pivot_table(
        index="depth_label",
        columns="width_label",
        values="seed",
        aggfunc="count",
        fill_value=0,
    ).reindex(index=DEPTH_LABEL_ORDER, columns=WIDTH_LABEL_ORDER)
)
print("done")

Cataloged 36 declared runs.
  graph_type: ['bus']
  encoder: ['gine']
  readout: ['mean']
  substation_edges: ['False']
  substation_nodes: ['False']
  configured_steps: ['15000000']
  gnn_out_dim matches gnn_hidden_dim for every config


width_label,h16,h32,h64,h128
depth_label,,,,
mp1,3,3,3,3
mp2,3,3,3,3
mp3,3,3,3,3


done


## Load the W&B histories

Only the declared `gs_s3dw` run names are selected. Runs that have not been
downloaded simply appear as missing rather than breaking the notebook, so this
works while the sweep is still filling in.

In [4]:
# Must happen before load_wandb_data(): full_history_to_long() reads this
# module-level whitelist at call time. wm is reloaded in the setup cell, so this
# does not leak into other notebooks.
if ACTOR_PARAM_METRIC not in wm.METRICS:
    wm.METRICS = [*wm.METRICS, ACTOR_PARAM_METRIC]
    print(f"Added {ACTOR_PARAM_METRIC} to the loaded metric whitelist.")

requested_run_names = config_catalog["run_name"].drop_duplicates().tolist()
requested_run_name_set = set(requested_run_names)
wm.configure_run_filter_from_names(requested_run_names)
data = wm.load_wandb_data(use_local_cache_only=USE_LOCAL_CACHE_ONLY)

runs_df = data.runs_df.copy()
history_df = data.history_df.copy()
if not history_df.empty:
    history_df = history_df[
        history_df["run_name"].astype(str).isin(requested_run_name_set)
    ].copy()
if "name" in runs_df:
    runs_df = runs_df[
        runs_df["name"].astype(str).isin(requested_run_name_set)
    ].copy()


def parse_optional_bool(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes", "y"}:
            return True
        if normalized in {"false", "0", "no", "n"}:
            return False
    return bool(value)


if "cuda" in runs_df and "name" in runs_df:
    runtime_backend = runs_df[["name", "cuda"]].copy()
    runtime_backend["runtime_cuda"] = runtime_backend["cuda"].map(
        parse_optional_bool
    )
    runtime_backend = (
        runtime_backend.dropna(subset=["runtime_cuda"])
        .drop_duplicates("name", keep="last")
        .rename(columns={"name": "run_name"})[["run_name", "runtime_cuda"]]
    )
    config_catalog = config_catalog.merge(
        runtime_backend, on="run_name", how="left"
    )
else:
    config_catalog["runtime_cuda"] = np.nan
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].where(
    config_catalog["runtime_cuda"].notna(), config_catalog["declared_cuda"]
)
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].astype(bool)
config_catalog["compute_backend"] = np.where(
    config_catalog["runtime_cuda"], "IZAR (GPU)", "JED (CPU)"
)

found_run_names = set(history_df.get("run_name", pd.Series(dtype=str)))
print(
    f"Loaded histories for {len(found_run_names)} / "
    f"{len(requested_run_names)} declared runs."
)
missing_run_names = sorted(requested_run_name_set - found_run_names)
if missing_run_names:
    print(f"Not downloaded ({len(missing_run_names)}):")
    for name in missing_run_names:
        print("  ", name)
print("done")

Added model/actor_params to the loaded metric whitelist.
Explicit run-name filter: 36 candidates
Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 12 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    12
History artifact setup: local_only=True, runs_df=12
[ 1/12] loading artifact cache: gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s3dw/runs/gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0__MAPPO_bus14_T_0_0__I__1785516702_27461/history.parquet in 0.1s
[ 2/12] loading artifact cache: gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s0
    loaded 361 rows, 113 columns from /User

/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py:626: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(pieces, ignore_index=True, sort=False)


[10/12] loading artifact cache: gs_s3dw_bus_n0_none_e0n0v0_mp3_h16_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s3dw/runs/gs_s3dw_bus_n0_none_e0n0v0_mp3_h16_s0__MAPPO_bus14_T_0_0__I__1785516702_21672/history.parquet in 0.0s
[11/12] loading artifact cache: gs_s3dw_bus_n0_none_e0n0v0_mp3_h32_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s3dw/runs/gs_s3dw_bus_n0_none_e0n0v0_mp3_h32_s0__MAPPO_bus14_T_0_0__I__1785516724_8641/history.parquet in 0.0s
[12/12] loading artifact cache: gs_s3dw_bus_n0_none_e0n0v0_mp3_h64_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s3dw/runs/gs_s3dw_bus_n0_none_e0n0v0_mp3_h64_s0__MAPPO_bus14_T_0_0__I__1785516792_30872/history.parquet in 0.0s
Finished history artifact loading in 0.2s
Loaded histories for 12 / 12 selected runs
Saved cache index: 

## Coverage and available seeds

`SEEDS_PER_ARCHITECTURE` drives the rest of the notebook: with one seed the
uncertainty bands and per-cell spread are suppressed, with more they are shown.

In [5]:
if history_df.empty:
    raise RuntimeError(
        "No history was loaded. Download the gs_s3dw runs first "
        "(analysis/download/download_wandb_group.py --group gs_s3dw), "
        "or set USE_LOCAL_CACHE_ONLY = False."
    )

observed_progress = (
    history_df.groupby("run_name", as_index=False)["step"]
    .max()
    .rename(columns={"step": "observed_steps"})
)
coverage = config_catalog.merge(observed_progress, on="run_name", how="left")
coverage["observed_steps_m"] = coverage["observed_steps"] / 1_000_000
coverage["completion_pct"] = (
    100 * coverage["observed_steps"] / coverage["configured_steps"]
)
coverage["has_history"] = coverage["observed_steps"].notna()

analysis_catalog = coverage[coverage["has_history"]].copy()

seeds_present = sorted(analysis_catalog["seed"].unique())
seed_counts = analysis_catalog.groupby("architecture")["seed"].nunique()
SEEDS_PER_ARCHITECTURE = int(seed_counts.max()) if len(seed_counts) else 0
SHOW_SEED_UNCERTAINTY = SEEDS_PER_ARCHITECTURE > 1
UNCERTAINTY = "std" if SHOW_SEED_UNCERTAINTY else None

print(f"Seeds present: {seeds_present}")
print(f"Max seeds per architecture: {SEEDS_PER_ARCHITECTURE}")
print(
    "Seed uncertainty bands: "
    + ("enabled" if SHOW_SEED_UNCERTAINTY else "disabled (single seed)")
)
if set(seed_counts.unique()) - {SEEDS_PER_ARCHITECTURE}:
    print("Uneven seed coverage across architectures:")
    display(seed_counts.rename("seeds").reset_index())

missing_architectures = sorted(
    set(ARCHITECTURE_ORDER) - set(analysis_catalog["architecture"])
)
if missing_architectures:
    print("Architectures with no downloaded run:", missing_architectures)

with pd.option_context("display.max_colwidth", None):
    display(
        coverage.sort_values(["gnn_layers", "gnn_hidden_dim", "seed"])[
            [
                "architecture",
                "seed",
                "run_name",
                "compute_backend",
                "observed_steps_m",
                "completion_pct",
                "has_history",
            ]
        ].round(2)
    )

progress_plot = px.bar(
    analysis_catalog.sort_values("observed_steps_m"),
    x="observed_steps_m",
    y="run_name",
    color="depth_label",
    orientation="h",
    hover_data=["architecture", "seed"],
    title="gs_s3dw run coverage",
    labels={
        "observed_steps_m": "Observed environment steps (millions)",
        "run_name": "Run",
    },
    height=max(500, 24 * len(analysis_catalog)),
)
progress_plot.add_vline(
    x=TARGET_BUDGET_STEPS / 1_000_000,
    line_dash="dash",
    annotation_text="15M target",
)
progress_plot.show()
print("done")

Seeds present: [0]
Max seeds per architecture: 1
Seed uncertainty bands: disabled (single seed)


,architecture,seed,run_name,compute_backend,observed_steps_m,completion_pct,has_history
3,mp1_h16,0,gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s0,JED (CPU),14.97,99.81,True
4,mp1_h16,1,gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s1,JED (CPU),NaN,NaN,False
5,mp1_h16,2,gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s2,JED (CPU),NaN,NaN,False
6,mp1_h32,0,gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s0,JED (CPU),14.97,99.81,True
7,mp1_h32,1,gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s1,JED (CPU),NaN,NaN,False
8,mp1_h32,2,gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s2,JED (CPU),NaN,NaN,False
9,mp1_h64,0,gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s0,JED (CPU),14.97,99.81,True
10,mp1_h64,1,gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s1,JED (CPU),NaN,NaN,False
11,mp1_h64,2,gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s2,JED (CPU),NaN,NaN,False
0,mp1_h128,0,gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0,JED (CPU),14.97,99.81,True


done


## Actor parameter counts

`model/actor_params` is logged by every run, so the screened architectures can
be placed on a real capacity axis instead of only the nominal depth/width codes.

In [6]:
param_history = history_df[history_df["metric"] == ACTOR_PARAM_METRIC]
if param_history.empty:
    print(f"{ACTOR_PARAM_METRIC} was not logged; skipping the capacity axis.")
    actor_params = pd.DataFrame(columns=["run_name", "actor_params"])
else:
    actor_params = (
        param_history.groupby("run_name", as_index=False)["value"]
        .max()
        .rename(columns={"value": "actor_params"})
    )
    actor_params["actor_params"] = actor_params["actor_params"].astype(int)

analysis_catalog = analysis_catalog.merge(
    actor_params, on="run_name", how="left"
)

if not actor_params.empty:
    param_table = (
        analysis_catalog.groupby(
            ["depth_label", "width_label"], as_index=False
        )["actor_params"]
        .first()
        .pivot(index="depth_label", columns="width_label", values="actor_params")
        .reindex(index=DEPTH_LABEL_ORDER, columns=WIDTH_LABEL_ORDER)
    )
    print("Actor parameter count per architecture:")
    display(param_table)
print("done")

Actor parameter count per architecture:


width_label,h16,h32,h64,h128
depth_label,,,,
mp1,83581,92333,114445,177101
mp2,84237,94669,123213,211021
mp3,84893,97005,131981,244941


done


## Full-test evaluation of each run's best checkpoint

This is the headline statistic: a deterministic evaluation of the best test
checkpoint over the complete held-out split. The run name is taken from the
`checkpoint` field inside each JSON rather than from the file name, because the
naming convention has differed between sweeps; the file name is then
cross-checked against it.

In [7]:
def best_checkpoint_run_name(record):
    checkpoint_stem = Path(record["checkpoint"]).stem
    prefix = "best_test_"
    if not checkpoint_stem.startswith(prefix):
        raise ValueError(
            f"Expected a best_test_ checkpoint, got {checkpoint_stem!r}."
        )
    return checkpoint_stem[len(prefix):]


# rglob: results may sit directly in outputs/full_test_eval or in a dated
# subfolder such as gs_s3dw_s0_full_test_20260803.
full_test_paths = sorted(FULL_TEST_EVAL_DIR.rglob(FULL_TEST_GLOB))
print(
    f"Found {len(full_test_paths)} full-test JSON file(s) under "
    f"{FULL_TEST_EVAL_DIR}"
)

full_test_rows = []
name_mismatches = []
for result_path in full_test_paths:
    with result_path.open("r", encoding="utf-8") as file:
        record = json.load(file)
    run_name = best_checkpoint_run_name(record)
    if run_name not in requested_run_name_set:
        continue
    stem = result_path.stem
    if stem not in {run_name, f"best_test_{run_name}"} and run_name not in stem:
        name_mismatches.append((result_path.name, run_name))
    full_test_rows.append(
        {
            "run_name": run_name,
            "best_eval_step": int(record["checkpoint_global_step"]),
            "best_eval_survival_pct": float(record["survival_percent"]),
            "best_eval_episodes": int(record["eval_episodes"]),
            "best_eval_split": str(record["split"]),
            "best_eval_deterministic": bool(record["deterministic_eval"]),
            "best_eval_all_chronics": bool(
                record.get("eval_all_split_chronics", False)
            ),
            "best_eval_json": str(result_path.relative_to(wm.TASK_DIR)),
        }
    )

if name_mismatches:
    print("File name does not contain its checkpoint run name:")
    for file_name, run_name in name_mismatches:
        print(f"   {file_name} -> {run_name}")

full_test_results = pd.DataFrame(full_test_rows)
if full_test_results.empty:
    raise RuntimeError(
        f"No gs_s3dw full-test JSON files found under {FULL_TEST_EVAL_DIR} "
        "(searched recursively)."
    )

duplicate_runs = full_test_results[
    full_test_results.duplicated("run_name", keep=False)
]
if not duplicate_runs.empty:
    raise ValueError(
        "Multiple full-test results found for: "
        + ", ".join(sorted(duplicate_runs["run_name"].unique()))
    )
if not full_test_results["best_eval_split"].eq("test").all():
    raise ValueError("At least one full-test result does not use split='test'.")
if not full_test_results["best_eval_deterministic"].all():
    raise ValueError("At least one full-test result is not deterministic.")
episode_counts = sorted(full_test_results["best_eval_episodes"].unique())
if episode_counts != [EXPECTED_FULL_TEST_EPISODES]:
    print(
        "WARNING: full-test episode counts are not all "
        f"{EXPECTED_FULL_TEST_EPISODES}: {episode_counts}"
    )

endpoint_df = analysis_catalog.merge(
    full_test_results, on="run_name", how="inner"
)
endpoint_df["best_eval_step_m"] = endpoint_df["best_eval_step"] / 1_000_000

evaluated = set(endpoint_df["run_name"])
downloaded_without_eval = sorted(set(analysis_catalog["run_name"]) - evaluated)
if downloaded_without_eval:
    print(f"Downloaded but not full-test evaluated ({len(downloaded_without_eval)}):")
    for name in downloaded_without_eval:
        print("  ", name)

print(
    f"Matched {len(endpoint_df)} deterministic best-checkpoint evaluations; "
    f"episodes per run: {episode_counts}"
)

with pd.option_context("display.max_colwidth", None):
    display(
        endpoint_df.sort_values(
            "best_eval_survival_pct", ascending=False
        )[
            [
                "architecture",
                "seed",
                "actor_params",
                "best_eval_step_m",
                "best_eval_episodes",
                "best_eval_survival_pct",
                "best_eval_json",
            ]
        ].round(2)
    )
print("done")

Found 12 full-test JSON file(s) under /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval
Matched 12 deterministic best-checkpoint evaluations; episodes per run: [201]


,architecture,seed,actor_params,best_eval_step_m,best_eval_episodes,best_eval_survival_pct,best_eval_json
8,mp3_h128,0,244941,0.17,201,100.00,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp3_h128_s0.json
1,mp1_h16,0,83581,14.85,201,98.68,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s0.json
2,mp1_h32,0,92333,10.70,201,97.07,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s0.json
4,mp2_h128,0,211021,14.10,201,96.66,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h128_s0.json
3,mp1_h64,0,114445,14.60,201,96.54,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s0.json
10,mp3_h32,0,97005,14.10,201,93.39,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp3_h32_s0.json
11,mp3_h64,0,131981,14.10,201,92.97,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp3_h64_s0.json
5,mp2_h16,0,84237,13.44,201,84.19,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h16_s0.json
0,mp1_h128,0,177101,10.20,201,84.18,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0.json
7,mp2_h64,0,123213,13.93,201,68.38,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h64_s0.json


done


## Training curves

In [8]:
survival_long = sc.extract_survival_curves(
    history_df,
    catalog=analysis_catalog,
    split="test",
    smooth=1,
)
if survival_long.empty:
    raise RuntimeError("No test episodic-survival history was found.")

print("metrics used:", sorted(survival_long["metric"].unique()))
print(f"{survival_long['run_name'].nunique()} runs with survival curves")

max_survival_steps = survival_long.groupby("run_name")["step"].max()
automatic_common_budget = int(max_survival_steps.min())
comparison_budget = int(
    COMPARISON_BUDGET_STEPS
    if COMPARISON_BUDGET_STEPS is not None
    else automatic_common_budget
)
print(
    "Comparison budget:",
    f"{comparison_budget / 1_000_000:.3f}M steps",
    "(automatic common budget)"
    if COMPARISON_BUDGET_STEPS is None
    else "(user selected)",
)
if COMPARISON_BUDGET_STEPS is None:
    print(f"  set by the shortest run: {max_survival_steps.idxmin()}")
print("done")

metrics used: ['test/charts/episodic_survival']
12 runs with survival curves
Comparison budget: 14.515M steps (automatic common budget)
  set by the shortest run: gs_s3dw_bus_n0_none_e0n0v0_mp2_h128_s0
done


### Aggregated survival curves

One curve per architecture. Colour encodes width and dash encodes depth, so the
two factors stay separable in a twelve-curve legend.

In [9]:
WIDTH_COLORS = {
    "h16": "#1f77b4",
    "h32": "#2ca02c",
    "h64": "#ff7f0e",
    "h128": "#d62728",
}
DEPTH_DASHES = {"mp1": "dot", "mp2": "solid", "mp3": "dash"}

architecture_colors = {
    f"mp{depth}_h{width}": WIDTH_COLORS[f"h{width}"]
    for depth in DEPTH_ORDER
    for width in WIDTH_ORDER
}
architecture_dashes = {
    f"mp{depth}_h{width}": DEPTH_DASHES[f"mp{depth}"]
    for depth in DEPTH_ORDER
    for width in WIDTH_ORDER
}

absolute_figure = sc.plot_survival_comparison(
    survival_long,
    group_by="architecture",
    label_by="architecture",
    smooth=SMOOTH_WINDOW,
    uncertainty=UNCERTAINTY,
    min_members=1,
    show_members=SHOW_SEED_UNCERTAINTY,
    colors=architecture_colors,
    dashes=architecture_dashes,
    highlight=[BASELINE_LABEL],
    budget_step=comparison_budget,
    title="gs_s3dw: test episodic survival by depth and width",
    width=1450,
    height=700,
)
absolute_figure.show()
print("done")

done


### Small multiples

One figure faceted by depth (width varies inside each panel) and one faceted by
width (depth varies inside each panel).

In [10]:
depth_facet_figure = sc.plot_survival_comparison(
    survival_long,
    group_by="architecture",
    label_by=lambda row: row["architecture"].split("_")[1],
    facet_by="depth_label",
    facet_order=DEPTH_LABEL_ORDER,
    smooth=SMOOTH_WINDOW,
    uncertainty=UNCERTAINTY,
    min_members=1,
    colors=architecture_colors,
    budget_step=comparison_budget,
    title="gs_s3dw: width within each depth",
    width=1450,
    height=520,
    ncols=3,
)
depth_facet_figure.show()

width_facet_figure = sc.plot_survival_comparison(
    survival_long,
    group_by="architecture",
    label_by=lambda row: row["architecture"].split("_")[0],
    facet_by="width_label",
    facet_order=WIDTH_LABEL_ORDER,
    smooth=SMOOTH_WINDOW,
    uncertainty=UNCERTAINTY,
    min_members=1,
    colors=architecture_colors,
    budget_step=comparison_budget,
    title="gs_s3dw: depth within each width",
    width=1450,
    height=520,
    ncols=4,
)
width_facet_figure.show()
print("done")

done


### Difference from the `mp2_h128` baseline

Each curve is an architecture minus the aggregated baseline at the same step, so
the baseline is the flat zero line.

> **Do not add `facet_by=` to a `comparison="difference"` call.**
> `survival_comparison._difference_from_baseline` inner-joins the baseline on
> `[facet_column, step]`, and `mp2_h128` exists in only one panel of any facet,
> so faceting silently drops the other panels instead of raising.

In [11]:
difference_figure = sc.plot_survival_comparison(
    survival_long,
    group_by="architecture",
    label_by="architecture",
    smooth=SMOOTH_WINDOW,
    uncertainty="ci95" if SHOW_SEED_UNCERTAINTY else None,
    min_members=1,
    comparison="difference",
    baseline={"architecture": BASELINE_LABEL},
    colors=architecture_colors,
    dashes=architecture_dashes,
    budget_step=comparison_budget,
    y_range=None,
    title=f"gs_s3dw: survival difference from the {BASELINE_LABEL} baseline",
    y_title="Survival difference from baseline (pp)",
    width=1450,
    height=700,
)
difference_figure.add_hline(y=0.0, line_dash="dot", line_color="#6b7280")
difference_figure.show()
print("done")

done


### One panel per architecture, each against the baseline

`S3DW_SUBPLOTS` is a plain dict, so panels can be removed, reordered, or added
by hand. Each entry accepts the usual selectors -- `prefix`, `name`, `runs`,
`contains`, `regex`, `run_dir` -- plus `label`, `color`, and `width`.

In [12]:
def s3dw_run_prefix(architecture):
    """Prefix shared by every seed of one architecture."""
    return f"{S3DW_RUN_PREFIX}_{architecture}_s"


S3DW_BASELINE_SPEC = {
    "prefix": s3dw_run_prefix(BASELINE_LABEL),
    "label": BASELINE_LABEL,
    "color": "#6b7280",
    "width": 4,
}

S3DW_SUBPLOTS = {}
for architecture in ARCHITECTURE_ORDER:
    if architecture == BASELINE_LABEL:
        continue
    S3DW_SUBPLOTS[f"{BASELINE_LABEL}  vs  {architecture}"] = [
        S3DW_BASELINE_SPEC,
        {
            "prefix": s3dw_run_prefix(architecture),
            "label": architecture,
            "color": architecture_colors.get(architecture, "#d62728"),
        },
    ]

print(f"{len(S3DW_SUBPLOTS)} panels requested")

S3DW_MEAN_GROUPS = {
    title: wm.resolve_named_plot_specs(run_specs, history=history_df)
    for title, run_specs in S3DW_SUBPLOTS.items()
}
# A panel with fewer than two resolved specs would show the baseline alone,
# which reads as a real comparison but is not one.
dropped = [title for title, specs in S3DW_MEAN_GROUPS.items() if len(specs) < 2]
S3DW_MEAN_GROUPS = {
    title: specs for title, specs in S3DW_MEAN_GROUPS.items() if len(specs) >= 2
}
if dropped:
    print(f"Dropped {len(dropped)} panel(s) with missing runs:")
    for title in dropped:
        print("  ", title)
print(f"{len(S3DW_MEAN_GROUPS)} panels plotted")

s3dw_architecture_fig = wm.plot_run_mean_groups(
    S3DW_MEAN_GROUPS,
    split="test",
    smooth=SMOOTH_WINDOW,
    title=f"gs_s3dw: each architecture against the {BASELINE_LABEL} baseline",
    title_font_size=26,
    subplot_title_font_size=20,
    ncols=4,
    subplot_height=470,
    width=2400,
    y_range=[0, 105],
    show_members=SHOW_SEED_UNCERTAINTY,
    show_std=SHOW_SEED_UNCERTAINTY,
    save_name="gs_s3dw_architecture_vs_baseline_subplots",
    history=history_df,
    horizontal_spacing=0.045,
    vertical_spacing=0.09,
    margin={"l": 50, "r": 20, "t": 90, "b": 45},
)
# Explicit show(): the cell ends with print("done"), so a bare expression here
# would not be auto-displayed.
s3dw_architecture_fig.show()
print("done")

11 panels requested
11 panels plotted
Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s3dw (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_s3dw_architecture_vs_baseline_subplots.html


done


## Depth × width heatmaps

Cells use the full-test survival of each run's best checkpoint. With more than
one seed per architecture the cell value is the mean across seeds.

In [13]:
HEATMAP_VALUE_COLUMN = "best_eval_survival_pct"
heatmap_endpoint_df = endpoint_df.dropna(
    subset=[HEATMAP_VALUE_COLUMN]
).copy()
excluded = len(endpoint_df) - len(heatmap_endpoint_df)
if excluded:
    print(f"Excluded {excluded} run(s) without a valid full-test result.")


def depth_width_table(frame, value_column, aggfunc="mean"):
    table = frame.pivot_table(
        index="depth_label",
        columns="width_label",
        values=value_column,
        aggfunc=aggfunc,
    )
    return table.reindex(
        index=DEPTH_LABEL_ORDER, columns=WIDTH_LABEL_ORDER
    ).rename_axis(index="depth", columns="width")


def show_depth_width_heatmap(
    table,
    title,
    color_label,
    colorscale="Viridis",
    zmin=None,
    zmax=None,
    zmid=None,
    text_format=".1f",
    source_df=None,
):
    if table.empty or table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None
    print(title)
    display(table.round(2))
    kwargs = dict(
        text_auto=text_format,
        aspect="auto",
        color_continuous_scale=colorscale,
        labels={
            "x": "Hidden width",
            "y": "Message-passing depth",
            "color": color_label,
        },
        title=title,
    )
    if zmin is not None:
        kwargs["zmin"] = zmin
    if zmax is not None:
        kwargs["zmax"] = zmax
    if zmid is not None:
        kwargs["color_continuous_midpoint"] = zmid
    figure = px.imshow(table, **kwargs)
    figure.show()
    if source_df is not None:
        detail = (
            source_df[
                [
                    "architecture",
                    "seed",
                    "actor_params",
                    "run_name",
                    "config_path",
                    "best_eval_step_m",
                    "best_eval_json",
                    HEATMAP_VALUE_COLUMN,
                ]
            ]
            .sort_values(["architecture", "seed"])
            .reset_index(drop=True)
        )
        print(f"Configs used for: {title}")
        with pd.option_context("display.max_colwidth", None):
            display(detail.round(2))
    return figure


absolute_table = depth_width_table(heatmap_endpoint_df, HEATMAP_VALUE_COLUMN)
show_depth_width_heatmap(
    absolute_table,
    "Depth × width — full-test survival of best checkpoint",
    "Survival (%)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    source_df=heatmap_endpoint_df,
)
print("done")

Depth × width — full-test survival of best checkpoint


width,h16,h32,h64,h128
depth,,,,
mp1,98.68,97.07,96.54,84.18
mp2,84.19,67.39,68.38,96.66
mp3,20.50,93.39,92.97,100.00


Configs used for: Depth × width — full-test survival of best checkpoint


,architecture,seed,actor_params,run_name,config_path,best_eval_step_m,best_eval_json,best_eval_survival_pct
0,mp1_h128,0,177101,gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0.toml,10.20,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h128_s0.json,84.18
1,mp1_h16,0,83581,gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s0.toml,14.85,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h16_s0.json,98.68
2,mp1_h32,0,92333,gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s0.toml,10.70,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h32_s0.json,97.07
3,mp1_h64,0,114445,gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s0.toml,14.60,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp1_h64_s0.json,96.54
4,mp2_h128,0,211021,gs_s3dw_bus_n0_none_e0n0v0_mp2_h128_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp2_h128_s0.toml,14.10,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h128_s0.json,96.66
5,mp2_h16,0,84237,gs_s3dw_bus_n0_none_e0n0v0_mp2_h16_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp2_h16_s0.toml,13.44,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h16_s0.json,84.19
6,mp2_h32,0,94669,gs_s3dw_bus_n0_none_e0n0v0_mp2_h32_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp2_h32_s0.toml,12.11,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h32_s0.json,67.39
7,mp2_h64,0,123213,gs_s3dw_bus_n0_none_e0n0v0_mp2_h64_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp2_h64_s0.toml,13.93,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp2_h64_s0.json,68.38
8,mp3_h128,0,244941,gs_s3dw_bus_n0_none_e0n0v0_mp3_h128_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp3_h128_s0.toml,0.17,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp3_h128_s0.json,100.00
9,mp3_h16,0,84893,gs_s3dw_bus_n0_none_e0n0v0_mp3_h16_s0,configs/gnn_graph_screening/stage3_depth_width/gs_s3dw_bus_n0_none_e0n0v0_mp3_h16_s0.toml,4.89,outputs/full_test_eval/gs_s3dw_s0_full_test_20260803/gs_s3dw_bus_n0_none_e0n0v0_mp3_h16_s0.json,20.50


done


In [14]:
baseline_depth_label = f"mp{BASELINE_DEPTH}"
baseline_width_label = f"h{BASELINE_WIDTH}"
baseline_value = absolute_table.loc[baseline_depth_label, baseline_width_label]
if pd.isna(baseline_value):
    raise ValueError(
        "The baseline cell has no value, so a relative heatmap is undefined."
    )
print(f"Baseline ({BASELINE_LABEL}) = {baseline_value:.2f}%")

delta_table = absolute_table - baseline_value
delta_limit = float(np.nanmax(np.abs(delta_table.to_numpy())))
show_depth_width_heatmap(
    delta_table,
    f"Difference from the {BASELINE_LABEL} baseline",
    "Survival difference (pp)",
    colorscale="RdBu",
    zmin=-delta_limit,
    zmax=delta_limit,
    zmid=0.0,
    text_format="+.1f",
)

show_depth_width_heatmap(
    depth_width_table(heatmap_endpoint_df, "run_name", aggfunc="nunique"),
    "Runs contributing to each cell",
    "Runs",
    colorscale="Blues",
    zmin=0,
    text_format="d",
)

if SHOW_SEED_UNCERTAINTY:
    show_depth_width_heatmap(
        depth_width_table(
            heatmap_endpoint_df, HEATMAP_VALUE_COLUMN, aggfunc="std"
        ),
        "Seed-to-seed standard deviation per cell",
        "Std across seeds (pp)",
        colorscale="Oranges",
        zmin=0,
    )
else:
    print(
        "Only one seed per architecture: the spread heatmap is skipped because "
        "a single run has no seed-to-seed standard deviation."
    )
print("done")

Baseline (mp2_h128) = 96.66%
Difference from the mp2_h128 baseline


width,h16,h32,h64,h128
depth,,,,
mp1,2.02,0.41,-0.12,-12.47
mp2,-12.47,-29.27,-28.28,0.00
mp3,-76.16,-3.27,-3.68,3.34


Runs contributing to each cell


width,h16,h32,h64,h128
depth,,,,
mp1,1,1,1,1
mp2,1,1,1,1
mp3,1,1,1,1


Only one seed per architecture: the spread heatmap is skipped because a single run has no seed-to-seed standard deviation.
done


### Heatmap with the individual seed values

Same cells and colour scale as above, but each cell also prints its per-seed
values underneath the mean. Seeds are labelled explicitly (`s0`, `s1`, `s2`)
rather than positionally, so a missing seed is visible instead of silently
shifting the order.

In [15]:
def heatmap_text_color(value, vmin, vmax):
    """White on the dark end of the scale, near-black on the light end."""
    if pd.isna(value):
        return "#111827"
    span = (vmax - vmin) or 1.0
    return "#f9fafb" if (value - vmin) / span < 0.55 else "#111827"


def show_heatmap_with_seeds(
    frame,
    value_column,
    index_column,
    column_column,
    index_order,
    column_order,
    title,
    x_label,
    y_label,
    color_label,
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    seed_column="seed",
    mean_font_size=22,
    seed_font_size=11,
    width=980,
):
    mean_table = frame.pivot_table(
        index=index_column,
        columns=column_column,
        values=value_column,
        aggfunc="mean",
    ).reindex(index=index_order, columns=column_order)
    if mean_table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None

    seed_labels = {}
    for (row_key, column_key), group in frame.groupby(
        [index_column, column_column]
    ):
        ordered = group.sort_values(seed_column)
        seed_labels[(row_key, column_key)] = "   ".join(
            f"s{int(seed)} {value:.1f}"
            for seed, value in zip(
                ordered[seed_column], ordered[value_column]
            )
        )

    figure = go.Figure(
        go.Heatmap(
            z=mean_table.to_numpy(),
            x=[str(value) for value in mean_table.columns],
            y=[str(value) for value in mean_table.index],
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            colorbar={"title": color_label},
            hovertemplate=(
                f"{y_label}: %{{y}}<br>{x_label}: %{{x}}<br>"
                f"mean {color_label}: %{{z:.2f}}<extra></extra>"
            ),
        )
    )
    for row_index, row_key in enumerate(mean_table.index):
        for column_index, column_key in enumerate(mean_table.columns):
            value = mean_table.iloc[row_index, column_index]
            if pd.isna(value):
                continue
            color = heatmap_text_color(value, zmin, zmax)
            figure.add_annotation(
                x=str(column_key),
                y=str(row_key),
                text=f"<b>{value:.1f}</b>",
                showarrow=False,
                yshift=15,
                font={"size": mean_font_size, "color": color},
            )
            label = seed_labels.get((row_key, column_key))
            if label:
                figure.add_annotation(
                    x=str(column_key),
                    y=str(row_key),
                    text=label,
                    showarrow=False,
                    yshift=-16,
                    font={"size": seed_font_size, "color": color},
                )

    figure.update_layout(
        title=title,
        xaxis_title=x_label,
        yaxis_title=y_label,
        width=width,
        height=150 * len(mean_table.index) + 200,
        # Match px.imshow, which puts the first row at the top.
        yaxis={"autorange": "reversed"},
    )
    figure.show()
    return mean_table


seeded_absolute_table = show_heatmap_with_seeds(
    heatmap_endpoint_df,
    HEATMAP_VALUE_COLUMN,
    "depth_label",
    "width_label",
    DEPTH_LABEL_ORDER,
    WIDTH_LABEL_ORDER,
    "Depth × width — cell mean with the individual seed values",
    "Hidden width",
    "Message-passing depth",
    "Survival (%)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
)
display(seeded_absolute_table.round(2))
print("done")

width_label,h16,h32,h64,h128
depth_label,,,,
mp1,98.68,97.07,96.54,84.18
mp2,84.19,67.39,68.38,96.66
mp3,20.50,93.39,92.97,100.00


done


## Survival against actor capacity

The depth/width grid exists to find a sweet spot, so plotting the full-test
result against the actual parameter count is the most direct read: if survival
plateaus or falls beyond some capacity, the largest architecture is not the one
to promote.

In [16]:
if "actor_params" not in endpoint_df or endpoint_df["actor_params"].isna().all():
    print("No actor parameter counts available; skipping the capacity plot.")
else:
    capacity_df = endpoint_df.dropna(
        subset=["actor_params", HEATMAP_VALUE_COLUMN]
    ).copy()
    capacity_figure = px.scatter(
        capacity_df.sort_values("actor_params"),
        x="actor_params",
        y=HEATMAP_VALUE_COLUMN,
        color="depth_label",
        symbol="width_label",
        category_orders={
            "depth_label": DEPTH_LABEL_ORDER,
            "width_label": WIDTH_LABEL_ORDER,
        },
        hover_data=["architecture", "seed", "run_name", "best_eval_step_m"],
        log_x=True,
        title=(
            "Full-test survival against actor parameter count "
            "(log scale)"
        ),
        labels={
            "actor_params": "Actor parameters",
            HEATMAP_VALUE_COLUMN: "Full-test survival (%)",
            "depth_label": "Depth",
            "width_label": "Width",
        },
        height=620,
        width=1100,
    )
    # Join the architectures of equal depth so the depth trend is readable.
    for depth_label in DEPTH_LABEL_ORDER:
        subset = capacity_df[capacity_df["depth_label"] == depth_label]
        if subset.empty:
            continue
        trend = (
            subset.groupby("actor_params", as_index=False)[HEATMAP_VALUE_COLUMN]
            .mean()
            .sort_values("actor_params")
        )
        capacity_figure.add_trace(
            go.Scatter(
                x=trend["actor_params"],
                y=trend[HEATMAP_VALUE_COLUMN],
                mode="lines",
                line={"width": 1.5, "dash": "dot"},
                name=f"{depth_label} trend",
                showlegend=False,
                hoverinfo="skip",
            )
        )
    capacity_figure.add_hline(
        y=baseline_value,
        line_dash="dash",
        line_color="#6b7280",
        annotation_text=f"{BASELINE_LABEL} baseline",
    )
    capacity_figure.show()
print("done")

done


## Architecture ranking and marginal effects

In [17]:
ranking = (
    heatmap_endpoint_df.groupby(
        ["architecture", "depth_label", "width_label"], as_index=False
    )
    .agg(
        mean_survival_pct=(HEATMAP_VALUE_COLUMN, "mean"),
        std_survival_pct=(HEATMAP_VALUE_COLUMN, "std"),
        min_survival_pct=(HEATMAP_VALUE_COLUMN, "min"),
        max_survival_pct=(HEATMAP_VALUE_COLUMN, "max"),
        seeds=("seed", "nunique"),
        actor_params=("actor_params", "first"),
    )
    .sort_values("mean_survival_pct", ascending=False)
)
ranking["mean_minus_baseline"] = ranking["mean_survival_pct"] - baseline_value
display(ranking.round(2))

rank_plot = px.bar(
    ranking.sort_values("mean_minus_baseline"),
    x="mean_minus_baseline",
    y="architecture",
    orientation="h",
    color="mean_minus_baseline",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    hover_data=["mean_survival_pct", "seeds", "actor_params"],
    title=f"Full-test survival relative to {BASELINE_LABEL}",
    labels={
        "mean_minus_baseline": "Difference from baseline (pp)",
        "architecture": "Architecture",
    },
    height=560,
)
rank_plot.add_vline(x=0.0, line_dash="dot")
rank_plot.show()
print("done")

,architecture,depth_label,width_label,mean_survival_pct,std_survival_pct,min_survival_pct,max_survival_pct,seeds,actor_params,mean_minus_baseline
8,mp3_h128,mp3,h128,100.00,NaN,100.00,100.00,1,244941,3.34
1,mp1_h16,mp1,h16,98.68,NaN,98.68,98.68,1,83581,2.02
2,mp1_h32,mp1,h32,97.07,NaN,97.07,97.07,1,92333,0.41
4,mp2_h128,mp2,h128,96.66,NaN,96.66,96.66,1,211021,0.00
3,mp1_h64,mp1,h64,96.54,NaN,96.54,96.54,1,114445,-0.12
10,mp3_h32,mp3,h32,93.39,NaN,93.39,93.39,1,97005,-3.27
11,mp3_h64,mp3,h64,92.97,NaN,92.97,92.97,1,131981,-3.68
5,mp2_h16,mp2,h16,84.19,NaN,84.19,84.19,1,84237,-12.47
0,mp1_h128,mp1,h128,84.18,NaN,84.18,84.18,1,177101,-12.47
7,mp2_h64,mp2,h64,68.38,NaN,68.38,68.38,1,123213,-28.28


done


## Convergence measured from the curves

The best-checkpoint statistic is a maximum over roughly 180 ten-episode
evaluations, so it rewards a lucky draw as much as a good policy. These three
measures use the whole curve instead of its largest point, and none of them can
be dominated by a single evaluation:

- **steps to a threshold** -- the first evaluation at which the smoothed curve
  reaches 60/80/90 %, i.e. how fast the run got there;
- **curve mean** -- the time-average of the smoothed curve over training, i.e.
  how good the policy was throughout, not just at its peak;
- **final-window mean** -- the mean of the last `FINAL_WINDOW_EVALS`
  evaluations, i.e. where training actually ended.

A run that never reaches a threshold gets `NaN` for it rather than the last
step, so a missing value reads as "never got there" instead of "got there at
the end".


In [18]:
# numpy renamed trapz to trapezoid in 2.0; keep both versions working.
_trapezoid = getattr(np, "trapezoid", np.trapz)

CONVERGENCE_THRESHOLDS = [60.0, 80.0, 90.0]
FINAL_WINDOW_EVALS = 10


def convergence_row(frame):
    """Threshold crossings, curve mean, and end-of-training mean for one run."""
    ordered = frame.sort_values("step")
    smoothed = (
        ordered["raw_survival_pct"]
        .rolling(SMOOTH_WINDOW, min_periods=1)
        .mean()
    )
    steps = ordered["step"].to_numpy()
    row = {
        "n_evals": len(ordered),
        # Time-average, so an uneven evaluation cadence cannot tilt the mean.
        "curve_mean_pct": float(
            _trapezoid(smoothed.to_numpy(), steps) / steps.max()
        )
        if steps.max() > 0
        else np.nan,
        "final_window_pct": float(smoothed.tail(FINAL_WINDOW_EVALS).mean()),
        "peak_pct": float(smoothed.max()),
    }
    for threshold in CONVERGENCE_THRESHOLDS:
        reached = steps[smoothed.to_numpy() >= threshold]
        row[f"steps_to_{int(threshold)}pct_m"] = (
            float(reached[0] / 1_000_000) if len(reached) else np.nan
        )
    return row


convergence = pd.DataFrame(
    [
        {"run_name": run_name, **convergence_row(frame)}
        for run_name, frame in survival_long.groupby("run_name", sort=False)
    ]
)
convergence = analysis_catalog.merge(convergence, on="run_name", how="inner")

threshold_columns = [
    f"steps_to_{int(threshold)}pct_m" for threshold in CONVERGENCE_THRESHOLDS
]
convergence_summary = (
    convergence.groupby(
        ["architecture", "depth_label", "width_label"], as_index=False
    )
    .agg(
        {
            **{column: "mean" for column in threshold_columns},
            "curve_mean_pct": "mean",
            "final_window_pct": "mean",
            "peak_pct": "mean",
        }
    )
    .sort_values("curve_mean_pct", ascending=False)
)

print(
    "Convergence from the training curves "
    f"(smoothing window {SMOOTH_WINDOW}, final window {FINAL_WINDOW_EVALS} evals):"
)
display(
    convergence_summary.set_index("architecture")[
        [*threshold_columns, "curve_mean_pct", "final_window_pct", "peak_pct"]
    ].round(2)
)

never_reached = convergence_summary.loc[
    convergence_summary["steps_to_90pct_m"].isna(), "architecture"
].tolist()
if never_reached:
    print("Never reached 90%:", ", ".join(never_reached))

# The endpoint statistic and the curves rank the architectures differently, and
# that disagreement is the point: it shows how much of the endpoint ranking is
# best-checkpoint selection rather than policy quality.
rank_compare = (
    ranking[["architecture", "mean_survival_pct"]]
    .merge(
        convergence_summary[["architecture", "curve_mean_pct"]],
        on="architecture",
    )
    .assign(
        endpoint_rank=lambda frame: frame["mean_survival_pct"]
        .rank(ascending=False)
        .astype(int),
        curve_rank=lambda frame: frame["curve_mean_pct"]
        .rank(ascending=False)
        .astype(int),
    )
    .sort_values("curve_rank")
)
rank_compare["rank_shift"] = (
    rank_compare["endpoint_rank"] - rank_compare["curve_rank"]
)
print("Endpoint ranking against curve ranking (positive = curves rank it higher):")
display(rank_compare.round(2))

convergence_figure = px.scatter(
    rank_compare,
    x="curve_mean_pct",
    y="mean_survival_pct",
    text="architecture",
    title="Best-checkpoint survival against curve mean",
    labels={
        "curve_mean_pct": "Curve mean over training (%)",
        "mean_survival_pct": "Full-test survival of best checkpoint (%)",
    },
    height=620,
    width=900,
)
convergence_figure.update_traces(textposition="top center")
convergence_figure.show()
print("done")


Convergence from the training curves (smoothing window 5, final window 10 evals):


,steps_to_60pct_m,steps_to_80pct_m,steps_to_90pct_m,curve_mean_pct,final_window_pct,peak_pct
architecture,,,,,,
mp2_h128,3.48,4.81,4.81,75.07,98.49,100.00
mp1_h32,4.81,5.97,6.14,73.31,91.50,97.34
mp1_h64,5.56,5.81,6.55,69.85,97.99,99.45
mp3_h32,3.32,6.14,6.30,69.82,95.25,97.03
mp1_h16,4.81,9.37,9.46,64.58,81.28,100.00
mp3_h64,3.82,4.73,11.78,63.92,92.83,97.67
mp1_h128,5.81,7.80,13.35,63.19,85.97,91.02
mp2_h64,3.15,NaN,NaN,56.47,69.11,77.36
mp2_h32,4.31,NaN,NaN,52.58,68.31,76.39


Never reached 90%: mp2_h64, mp2_h32, mp3_h128, mp2_h16, mp3_h16
Endpoint ranking against curve ranking (positive = curves rank it higher):


,architecture,mean_survival_pct,curve_mean_pct,endpoint_rank,curve_rank,rank_shift
3,mp2_h128,96.66,75.07,4,1,3
2,mp1_h32,97.07,73.31,3,2,1
4,mp1_h64,96.54,69.85,5,3,2
5,mp3_h32,93.39,69.82,6,4,2
1,mp1_h16,98.68,64.58,2,5,-3
6,mp3_h64,92.97,63.92,7,6,1
8,mp1_h128,84.18,63.19,9,7,2
9,mp2_h64,68.38,56.47,10,8,2
10,mp2_h32,67.39,52.58,11,9,2
0,mp3_h128,100.00,50.94,1,10,-9


done


In [19]:
for factor, label, order in [
    ("depth_label", "Message-passing depth", DEPTH_LABEL_ORDER),
    ("width_label", "Hidden width", WIDTH_LABEL_ORDER),
]:
    summary = (
        heatmap_endpoint_df.groupby(factor, as_index=False)
        .agg(
            mean_survival_pct=(HEATMAP_VALUE_COLUMN, "mean"),
            std_survival_pct=(HEATMAP_VALUE_COLUMN, "std"),
            min_survival_pct=(HEATMAP_VALUE_COLUMN, "min"),
            max_survival_pct=(HEATMAP_VALUE_COLUMN, "max"),
            runs=("run_name", "nunique"),
        )
        .set_index(factor)
        .reindex(order)
        .reset_index()
    )
    print(f"{label} (averaged over the other factor):")
    display(summary.round(2))
print("done")

Message-passing depth (averaged over the other factor):


,depth_label,mean_survival_pct,std_survival_pct,min_survival_pct,max_survival_pct,runs
0,mp1,94.12,6.68,84.18,98.68,4
1,mp2,79.15,13.98,67.39,96.66,4
2,mp3,76.71,37.62,20.50,100.00,4


Hidden width (averaged over the other factor):


,width_label,mean_survival_pct,std_survival_pct,min_survival_pct,max_survival_pct,runs
0,h16,67.79,41.59,20.50,98.68,3
1,h32,85.95,16.18,67.39,97.07,3
2,h64,85.96,15.33,68.38,96.54,3
3,h128,93.61,8.34,84.18,100.00,3


done


## Optional CSV export

In [20]:
EXPORT_TABLES = False

if EXPORT_TABLES:
    export_dir = wm.TASK_DIR / "outputs" / "gs_s3dw_depth_width_summary"
    export_dir.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(export_dir / "coverage.csv", index=False)
    endpoint_df.to_csv(export_dir / "run_endpoints.csv", index=False)
    ranking.to_csv(export_dir / "architecture_ranking.csv", index=False)
    absolute_table.to_csv(export_dir / "heatmap_absolute.csv")
    delta_table.to_csv(export_dir / "heatmap_baseline_delta.csv")
    print("Saved tables under", export_dir)
else:
    print("Set EXPORT_TABLES = True to write CSVs.")
print("done")

Set EXPORT_TABLES = True to write CSVs.
done
